## Robustness analysis

- Multiple reruns with varying architecture settings
- Test latent representation disentanglement & trajectory stability

In [ ]:
import os
import gc
import sys
import time

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import pyro
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from IPython.display import display

sns.set_context('paper')
rcParams.update({'font.family': 'Liberation Sans'})
rcParams.update({'font.size': 12})
rcParams.update({'figure.dpi': 180})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, plot, utils, trajectory, metrics
import vgae, configs, dataset
from importlib import reload

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

Load datasets:

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
model_outdir = '../results/liver/robustness/'
sample_id = 'NIH_F5'

adata_xenium = IO.load_xenium(os.path.join(xenium_path, sample_id))
adata_desi = sc.read_h5ad(os.path.join(desi_path, sample_id+'.h5'))
adata_xenium, adata_desi = IO.filter_cells(adata_xenium, adata_desi, by='map')

adata_xenium.obs['leiden'], categories = adata_xenium.obs.cell_type.factorize()
categories = categories.values


### Training

Multiple reruns:

In [ ]:
# Dataset specs
n_subgraphs = 16
k = 20
r = 50
sigma = 20

graph_data = dataset.HeteroDataset(
    adatas_ref=adata_xenium, 
    adatas_query=adata_desi,
    n_subgraphs=n_subgraphs, 
    k=k,
    r=r,
    is_weighted=True,
    sigma=sigma,
    verbose=True
)

train_data, val_data = random_split(graph_data, [0.7, 0.3])
train_dl, val_dl = DataLoader(train_data, shuffle=True), DataLoader(val_data)

# Model parameters
n_hidden = 32
n_latent = 6

# Training parameters
n_epochs = 500
lr = 1e-2
patience = 50

# Training & Inference
train_configs = configs.set_train_configs(
    n_epochs=n_epochs, lr=lr, patience=patience, gamma=1., anneal=False,
    device=torch.device('cuda'),
)


- Runs w/ approx. GP + Additive decoder
- Runs w/ only approx. GP
- Runs w/ non of the designs

In [ ]:
# Test: whether longer training helps with stability??
n_trials = 20
seeds = [0, 42, 1, 1234, 123, 10, 3, 12345, 2, 5, 100, 4, 13, 11, 1999, 113, 8, 23, 211, 9]
model_outdir = '../results/liver/robustness/'

for i in range(n_trials):
    pyro.clear_param_store()
    torch.cuda.empty_cache()
    gc.collect()

    model_configs = configs.set_model_configs(
        c_in=adata_xenium.shape[1], c_aux=adata_desi.shape[1],  # ref & query dims
        c_hidden=n_hidden, c_latent=n_latent, act=nn.SiLU(), dropout=0., seed=seeds[i],
        ref=graph_data.ref, query=graph_data.query, 
        num_clusters=graph_data.num_clusters,
        gene_symbols=adata_xenium.var_names,
    ) 

    model = vgae.HeteroVGAE(model_configs, device=torch.device('cuda'))
    model.fit(train_configs, train_dl=train_dl, val_dl=val_dl, DEBUG=True)
    
    # torch.save(model.state_dict(), os.path.join(model_outdir, 'lynx_{0}_run_{1}.pt'.format(n_latent, i)))
    # torch.save(model.state_dict(), os.path.join(model_outdir, 'lynx_{0}_wo_addec_run_{1}.pt'.format(n_latent, i)))
    torch.save(model.state_dict(), os.path.join(model_outdir, 'lynx_{0}_mlp_run_{1}.pt'.format(n_latent, i)))
    
    del model

### Evaluation

In [ ]:
model_outdir = '../results/liver/robustness/'

n_trials = 20

n_hidden = 32
n_latent = 6
n_subgraphs = 16
k = 20
r = 50
sigma = 20

graph_data = dataset.HeteroDataset(
    adatas_ref=adata_xenium, 
    adatas_query=adata_desi,
    n_subgraphs=n_subgraphs, 
    k=k,
    r=r,
    is_weighted=True,
    sigma=sigma,
    verbose=True
)

model_configs = configs.set_model_configs(
    c_in=adata_xenium.shape[1], c_aux=adata_desi.shape[1],  # ref & query dims
    c_hidden=n_hidden, c_latent=n_latent, act=nn.SiLU(), dropout=0.,
    ref=graph_data.ref, query=graph_data.query, 
    num_clusters=graph_data.num_clusters,
    gene_symbols=adata_xenium.var_names,
) 


In [ ]:
model_outputs = []

for i in range(n_trials):
    model = vgae.HeteroVGAE(model_configs, device=torch.device('cpu'))
    # model.load_state_dict(torch.load(os.path.join(model_outdir, 'lynx_{0}_run_{1}.pt'.format(n_latent, i))))
    # model.load_state_dict(torch.load(os.path.join(model_outdir, 'lynx_{0}_wo_addec_run_{1}.pt'.format(n_latent, i))))
    model.load_state_dict(torch.load(os.path.join(model_outdir, 'lynx_{0}_mlp_run_{1}.pt'.format(n_latent, i))))
    
    res = model.evaluate(adata_xenium, adata_desi, graph_data=graph_data, device=torch.device('cpu'))
    model_outputs.append(res)

    del model
    gc.collect()

Task (1). stability of trajectory `(t)`

In [ ]:
desi_ts = []
xenium_ts = []

for i in range(n_trials):
    res = model_outputs[i]
    adata_desi.obsm['X_z'] = res.qzu
    adata_xenium.obsm['X_z'] = res.qzx

    trajectory.compute_trajectory(adata_desi, root_marker='Taurine ')
    trajectory.compute_trajectory(adata_xenium, root_marker='DPT')
    
    desi_ts.append(adata_desi.obs['t'].values)
    xenium_ts.append(adata_xenium.obs['t'].values)

    del res, adata_desi.obsm['X_z'], adata_xenium.obsm['X_z']

xenium_t_df = pd.DataFrame(np.vstack(xenium_ts).T)
desi_t_df = pd.DataFrame(np.vstack(desi_ts).T)

In [ ]:
indices = np.argsort(xenium_t_df.mean(1).to_numpy())
sample_mean = np.sort(xenium_t_df.mean(1).to_numpy())
sample_std = xenium_t_df.std(1).to_numpy()[indices]
xx = np.linspace(0, 1, adata_xenium.shape[0])

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(xx, sample_mean, '-', color='purple', lw=3, label='Mean')
ax.fill_between(xx, sample_mean - sample_std, sample_mean + sample_std, alpha=0.3, label='±1 STD')
ax.set_xlabel("Index")
ax.set_ylabel("Value")
ax.set_title(r"Xenium spatial trajectory $(t)$ across runs - LYNX", fontsize=10)
ax.legend()

ax.spines[['right', 'top']].set_visible(False)
plt.show()

indices = np.argsort(desi_t_df.mean(1).to_numpy())
sample_mean = np.sort(desi_t_df.mean(1).to_numpy())
sample_std = desi_t_df.std(1).to_numpy()[indices]
xx = np.linspace(0, 1, adata_desi.shape[0])

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(xx, sample_mean, '-', color='purple', lw=3, label='Mean')
ax.fill_between(xx, sample_mean - sample_std, sample_mean + sample_std, alpha=0.3, label='±1 STD')
ax.set_xlabel("Index")
ax.set_ylabel("Value")
ax.set_title(r"DESI spatial trajectory $(t)$ across runs - LYNX", fontsize=10)
ax.legend()

ax.spines[['right', 'top']].set_visible(False)
plt.show()

del indices, sample_mean, sample_std

In [ ]:
df_melt = xenium_t_df.melt(var_name='Run', value_name='Value')
df_melt['Index'] = df_melt.groupby('Run').cumcount()
df_melt.head()

rand_indices = np.sort(np.random.choice(np.arange(adata_xenium.shape[0]), 20, replace=False))
df_melt_ss = df_melt.loc[df_melt.Index.isin(rand_indices), :]
df_melt_ss.shape

fig, ax = plt.subplots(figsize=(20, 5))
sns.violinplot(data=df_melt_ss, x='Index', y='Value', palette=sns.color_palette(), alpha=0.5, ax=ax)

sns.swarmplot(data=df_melt_ss, x='Index', y='Value', color='black', alpha=0.2, ax=ax)
plt.title("Violin plot per Index Across Runs", fontsize=20)

ax.set_ylim([-.05, 1.05])
ax.spines[['right', 'top']].set_visible(False)
plt.show()

Task (2). Latent representation disentanglement

In [ ]:
for i in range(n_trials):
    res = model_outputs[i]
    plot.disp_factor_corr(res.qzu)
    del res

In [ ]:
# Off-diagonal correlations
from scipy.special import comb

offdiag_corrs = []
for i in range(n_trials):
    z_corr = np.corrcoef(model_outputs[i].qzu.T)
    offdiag_corrs.append(np.abs(np.tril(z_corr, k=-1)).sum() / comb(z_corr.shape[0], 2))

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
sns.swarmplot(x=offdiag_corrs, alpha=0.5, color='black', ax=ax)
sns.violinplot(x=offdiag_corrs, alpha=0.3, ax=ax)
ax.set_xlabel(r'$R$')
ax.set_title('Latent factor off-diagonal correlations')
ax.spines[['right', 'top']].set_visible(False)
ax.set_yticks([])
plt.show()

del z_corr

In [ ]:
# Check MCC (true disentanglement score)
from scipy.optimize import linear_sum_assignment

def mean_corr_coef_np(x, y):
    """
    # Reference: https://github.com/siamakz/iVAE/blob/master/lib/metrics.py
    """
    d = x.shape[1]
    cc = np.abs(np.corrcoef(x, y, rowvar=False)[:d, d:])
    score = cc[linear_sum_assignment(-1 * cc)].mean()
    return score


mccs = []
baseline_ccs = []
for i in range(n_trials-1):
    for j in range(i+1, n_trials):
        mcc = mean_corr_coef_np(model_outputs[i].qzu, model_outputs[j].qzu)
        mccs.append(mcc)
        baseline_cc = np.abs(np.corrcoef(model_outputs[i].qzu.T, model_outputs[j].qzu.T)[:n_latent, n_latent:]).mean()
        baseline_ccs.append(baseline_cc)
del i, j, mcc, baseline_cc


mcc_df = pd.DataFrame({
    'MCC': mccs + baseline_ccs,
    'Category': ['permutation']*len(mccs) + ['w.o. permutation']*len(mccs)
})

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
sns.violinplot(data=mcc_df, x='Category', y='MCC', palette=sns.color_palette("pastel"))
ax.spines[['right', 'top']].set_visible(False)
ax.set_title('Latent-factor correspondance across runs - LYNX')
plt.show()

Comparison across settings:

In [ ]:
%reload_ext autoreload

In [ ]:
results_mlp = []
for i in range(n_trials):
    model_mlp = vgae.HeteroVGAE(model_configs, device=torch.device('cpu'))
    model_mlp.load_state_dict(torch.load(os.path.join(model_outdir, 'lynx_{0}_mlp_run_{1}.pt'.format(n_latent, i))))
    results_mlp.append(
        model_mlp.evaluate(adata_xenium, adata_desi, graph_data=graph_data, device=torch.device('cpu'))
    )
    del model_mlp
gc.collect()


In [ ]:
results_sp = []
for i in range(n_trials):
    model_sp = vgae.HeteroVGAE(model_configs, device=torch.device('cpu'))
    model_sp.load_state_dict(torch.load(os.path.join(model_outdir, 'lynx_{0}_wo_addec_run_{1}.pt'.format(n_latent, i))))
    results_sp.append(
        model_sp.evaluate(adata_xenium, adata_desi, graph_data=graph_data, device=torch.device('cpu'))
    )
    del model_sp
gc.collect()

In [ ]:
results_sp_addec = []
for i in range(n_trials):
    model_sp_addec = vgae.HeteroVGAE(model_configs, device=torch.device('cpu'))
    model_sp_addec.load_state_dict(torch.load(os.path.join(model_outdir, 'lynx_{0}_run_{1}.pt'.format(n_latent, i))))
    results_sp_addec.append(
        model_sp_addec.evaluate(adata_xenium, adata_desi, graph_data=graph_data, device=torch.device('cpu'))
    )
    del model_sp_addec
gc.collect()

In [ ]:
from scipy.special import comb

offdiag_zcorrs_mlp = []
offdiag_zcorrs_sp = []
offdiag_zcorrs_sp_addec = []

for i in range(n_trials):
    z_corr = np.corrcoef(results_mlp[i].qzu.T)
    offdiag_zcorrs_mlp.append(np.abs(np.tril(z_corr, k=-1)).sum() / comb(z_corr.shape[0], 2))

    z_corr = np.corrcoef(results_sp[i].qzu.T)
    offdiag_zcorrs_sp.append(np.abs(np.tril(z_corr, k=-1)).sum() / comb(z_corr.shape[0], 2))

    z_corr = np.corrcoef(results_sp_addec[i].qzu.T)
    offdiag_zcorrs_sp_addec.append(np.abs(np.tril(z_corr, k=-1)).sum() / comb(z_corr.shape[0], 2))

latent_corr_df = pd.DataFrame({
    'Score': offdiag_zcorrs_mlp + offdiag_zcorrs_sp + offdiag_zcorrs_sp_addec,
    'Category': ['MLPs']*n_trials + ['Structural prior']*n_trials + ['Structural priors & Additive decoder']*n_trials
})


fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
sns.swarmplot(data=latent_corr_df, x='Score', y='Category', alpha=0.5, color='black', ax=ax)
sns.violinplot(data=latent_corr_df, x='Score', y='Category', alpha=0.3, ax=ax)
ax.set_xlabel(r'$R$')
ax.set_title('Latent factor off-diagonal correlations')
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del z_corr, latent_corr_df

Task (3). Average trajectory inference accuracy:

In [ ]:
ts_mlp = []
ts_sp = []
ts_sp_addec = []

for i in range(n_trials):
    res = results_mlp[i]
    adata_xenium.obsm['X_z'] = res.qzx
    trajectory.compute_trajectory(adata_xenium, root_marker='DPT')
    ts_mlp.append(adata_xenium.obs['t'].values)
    del res, adata_xenium.obsm['X_z']

    res = results_sp[i]
    adata_xenium.obsm['X_z'] = res.qzx
    trajectory.compute_trajectory(adata_xenium, root_marker='DPT')
    ts_sp.append(adata_xenium.obs['t'].values)
    del res, adata_xenium.obsm['X_z']

    res = results_sp_addec[i]
    adata_xenium.obsm['X_z'] = res.qzx
    trajectory.compute_trajectory(adata_xenium, root_marker='DPT')
    ts_sp_addec.append(adata_xenium.obs['t'].values)
    del res, adata_xenium.obsm['X_z']


In [ ]:
# Correlation w/ 'silver-standard' trajectory
from scipy.stats import pearsonr
from statannotations.Annotator import Annotator

t_gt = np.load('../results/liver/ablation/antibody_gamma.npy')
corrs_mlp = []
corrs_sp = []
corrs_sp_addec = []

for i in range(n_trials):
    corrs_mlp.append(pearsonr(t_gt, ts_mlp[i])[0])
    corrs_sp.append(pearsonr(t_gt, ts_sp[i])[0])
    corrs_sp_addec.append(pearsonr(t_gt, ts_sp_addec[i])[0])

traj_corr_df = pd.DataFrame({
    'Score': corrs_mlp + corrs_sp + corrs_sp_addec,
    'Category': ['MLPs']*n_trials + ['Structural prior']*n_trials + ['Structural priors & Additive decoder']*n_trials
})

fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
ax = sns.violinplot(data=traj_corr_df, x='Category', y='Score', palette=sns.color_palette(), alpha=0.5, ax=ax)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel(r"$R$")
ax.set_title('Spatial Trajectory\n LYNX vs. Antibody')

pairs = [
    ('MLPs', 'Structural prior'), 
    ('MLPs', 'Structural priors & Additive decoder'), 
    ('Structural prior', 'Structural priors & Additive decoder')
]
annotator = Annotator(ax, pairs, data=traj_corr_df, x='Category', y='Score')
annotator.configure(test='Mann-Whitney', text_format='full')
annotator.apply_and_annotate()

plt.show()

del traj_corr_df


In [ ]:
# AP against individual antibody channels

# Load antibody images
ab_path = '../data/integrated/antibody/'

adata_ab = IO.load_ab_stain(
    os.path.join(ab_path, sample_id+'.ome.tif'),
    adata_ref=adata_xenium
)

# Normalize to [0, 1] per channel
scaled_chans = np.zeros_like(adata_ab.X)
for i, chan in enumerate(adata_ab.X.T):
    chan = (chan-chan.min()) / (chan.max()-chan.min())
    scaled_chans[:, i] = chan
adata_ab.X = scaled_chans

ab_dict = {
    'Opal 690-GS': 'Central Vein',
    'Opal 780-CYP3A4': 'Peri-central',
    'Opal 570-ASS1': 'Peri-portal',
    'Opal 520-Col1': 'Portal Vein'
}
ab_labels = list(ab_dict.keys())

y_gs = (adata_ab[:, ab_labels[0]].X > metrics.get_antibody_threshold(adata_ab, ab_labels[0])).squeeze().astype(np.uint8)
y_cyp = (adata_ab[:, ab_labels[1]].X > metrics.get_antibody_threshold(adata_ab, ab_labels[1])).squeeze().astype(np.uint8)
y_ass = (adata_ab[:, ab_labels[2]].X < metrics.get_antibody_threshold(adata_ab, ab_labels[2])).squeeze().astype(np.uint8)
y_col1 = (adata_ab[:, ab_labels[3]].X < metrics.get_antibody_threshold(adata_ab, ab_labels[3])).squeeze().astype(np.uint8)

y_antibodies = [y_gs, y_cyp, y_ass, y_col1]

In [ ]:
ap_mlp = []
ap_sp = []
ap_sp_addec = []

for i in range(n_trials):
    ap_mlp.extend(metrics.compute_ap(ts_mlp[i], y_antibodies).tolist())
    ap_sp.extend(metrics.compute_ap(ts_sp[i], y_antibodies).tolist())
    ap_sp_addec.extend(metrics.compute_ap(ts_sp_addec[i], y_antibodies).tolist())

n_aps = len(ap_mlp)

                       
ap_df = pd.DataFrame({
    'Score': ap_mlp + ap_sp + ap_sp_addec,
    'Category': ['MLPs']*n_aps + ['Structural prior']*n_aps + ['Structural priors & Additive decoder']*n_aps
})

fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
ax = sns.boxplot(data=ap_df, x='Category', y='Score', 
                 palette=sns.color_palette(), boxprops=dict(alpha=.5), ax=ax)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel(r"AP")
ax.set_title('Average Precision against\n individual Antibody images')

pairs = [
    ('MLPs', 'Structural prior'), 
    ('MLPs', 'Structural priors & Additive decoder'), 
    ('Structural prior', 'Structural priors & Additive decoder')
]
annotator = Annotator(ax, pairs, data=ap_df, x='Category', y='Score')
annotator.configure(test='Mann-Whitney', text_format='full')
annotator.apply_and_annotate()

plt.show()
                       
